In [ ]:
# IMDB Dataset
# IMDB.com 사이트에 있는 영화, 드라마 리뷰(영어)를 
# 우리가 모델로 학습할 수 있도록 이미 단어사전도 만들고, 숫자로 다 바꿔서
# 우리한테 제공하는 데이터셋
# Tensorflow를 이용해서 해당 내용을 구현해 보았어요!
# PyTorch로 구현해 보아요!

In [1]:
%reset -f
import numpy as np
import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter

2025-07-08 09:45:21.596507: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-08 09:45:21.733139: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751935521.790309     516 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751935521.806604     516 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751935521.917906     516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# PyTorch는 직접 데이터와 모델을 GPU Memory에 이동시켜야 해요!
# 항상 처음에 PyTorch환경이 GPU를 사용할 수 있는 환경인지 확인!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
# 데이터 로딩
# 우리가 사용하는 데이터는 순차데이터(시계열데이터)예요. timestep이라는 개념이 들어가 있어요!
# 자연어를 처리하려고 해요! 각각의 단어( = token = timestep )
# 이런 단어(toekn, timestep)의 연속이 우리의 sample이 되요!
# 원래는 vocabulary(단어사전)이라는 것부터 만들어야 해요! 그런데 IMDB Dataset은
# 이 과정이 이미 진행되어 있어요! 편하게 데이터를 불러다 사용할 수 있어요!
(x_data_train, y_data_train), (x_data_test, y_data_test) = \
imdb.load_data(num_words=500)

x_data_train, x_data_val, y_data_train, y_data_val = \
train_test_split(x_data_train,
                 y_data_train,
                 test_size=0.2,
                 stratify=y_data_train)

In [5]:
# 데이터 전처리
# 1. 길이를 맞춰야 해요!(리뷰길이가 제각각인데 동일 길이로 맞춰야 해요!)
# Padding처리를 통해서 길이를 맞춰요!
x_data_train_seq = pad_sequences(x_data_train,
                                 maxlen=100)
x_data_val_seq = pad_sequences(x_data_val,
                               maxlen=100)
# 2. one-hot 인코딩으로 처리(token값들은 값의양이 아니예요. 분류값이예요!)
x_data_train_onehot = to_categorical(x_data_train_seq,
                                     num_classes=500)
x_data_val_onehot = to_categorical(x_data_val_seq,
                                   num_classes=500)

In [6]:
# PyTorch에서 사용하는 Tensor형태로 변환
# GPU를 사용하기 위해서 데이터를 이동
x_train_tensor = torch.FloatTensor(x_data_train_onehot).to(device)
y_train_tensor = torch.FloatTensor(y_data_train).to(device)

x_val_tensor = torch.FloatTensor(x_data_val_onehot).to(device)
y_val_tensor = torch.FloatTensor(y_data_val).to(device)

In [7]:
# 이렇게 만든 데이터를 조금 편하게 사용해 볼꺼예요!
# batch처리하기 편하도록 Dataloader를 이용할꺼예요!
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset,
                          batch_size=64,
                          shuffle=True)
val_loader = DataLoader(val_dataset,
                        batch_size=64)

In [8]:
# 데이터 처리가 끝났으니 Model을 구현해 보면 되요!
# 1. class를 정의해야 해요! -> 특정 class를 상속해서 만들어야 해요!
# 2. class내에 우리가 사용할 layer를 속성으로 정의해야 해요!
#    => __init__()안에서 구현
# 3. 순전파 기능을 함수로 구현해야 해요!
#    => forward() 안에서 구현.
class SimpleRNNModel(nn.Module):   # nn.Module를 상속해서 class를 정의
    def __init__(self, input_size, hidden_size):
        # input_size : RNN Model의 input차원 => vacabulary의 크기
        # hidden_size : RNN Layer안의 neuron(node)수
        super().__init__()   # 상위 class의 초기화 함수 호출
        # PyTorch가 제공하는 RNN Layer
        self.rnn = nn.RNN(input_size=input_size,
                          hidden_size=hidden_size,
                          batch_first=True)
        # batch_first=True는 어떤 의미인가요?
        # 기본적인 형태는 (seq_len, batch_size, input_size) 이거인데
        # (batch_size, seq_len, input_size) 이 형태가 필요해요!
        self.fc = nn.Linear(hidden_size, 1)  # hidden_size : 입력데이터 개수
                                             # 1 : 출력 데이터 개수 
    def forward(self, x):
        # 데이터를 어떻게 전달해서 순전파 연산을 수행할건지를 기술
        out, _ = self.rnn(x)
        # out : 모든 시점의 hidden state
        out = out[:, -1, :]   # 마지막 시점의 hidden state
        out = self.fc(out)
        return torch.sigmoid(out)  # 최종적인 순전파 결과값.

# 이렇게 기능이 정의된 모델 클래스로부터
# 실제로 사용할 수 있는 모델을 만들어요!
model = SimpleRNNModel(input_size=500, 
                       hidden_size=8).to(device)

In [10]:
# Loss, Optimizer가 있어야 해요!
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),
                       lr=1e-3)
# TensorBoard도 처리해야 하니
log_dir = './log/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
writer = SummaryWriter(log_dir=log_dir)

In [12]:
# 학습
# 안타깝게도.. fit()함수를 이용하지 않아요!
# 직접 loop를 돌면서 수동으로 처리!
# 참고로 lightning은 fit()을 이용해서 학습을 진행!

# 100 epochs를 수행할꺼예요!
for epoch in range(20):
    # 학습
    model.train()  # pytorch에서 모델 학습하기 전에는 반드시 이 함수를 호출
                   # 이 작업을 수행해야 나중에 역전파를 진행할 수 있어요!
    total_loss = 0.0
    total_acc = 0.0

    for x_batch, y_batch in train_loader:

        optimizer.zero_grad()
        outputs = model(x_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()   # backpropagation
        optimizer.step()  # weight값을 갱신

        total_loss += loss.item() * 64
        total_acc += ((outputs > 0.5).float() == y_batch).sum().item()

    train_loss = total_loss / len(train_loader.dataset)  # 전체 데이터 개수로 나눠줘요!
    train_acc =  total_acc / len(train_loader.dataset)   # 정확도가 나와요!

    # 검증
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            outputs = model(x_batch).squeeze()
            loss = criterion(outputs,y_batch)

            val_loss += loss.item() * 64
            val_acc += ((outputs > 0.5).float() == y_batch).sum().item()
            
        val_loss /= len(val_loader.dataset)
        val_acc /= len(val_loader.dataset)

    print(f'Epochs {epoch} Train Accuracy : {train_acc}')
    print(f'Epochs {epoch} Validation Accuracy : {val_acc}')

Epochs 0 Train Accuracy : 0.77345
Epochs 0 Validation Accuracy : 0.7344
Epochs 1 Train Accuracy : 0.76325
Epochs 1 Validation Accuracy : 0.7554
Epochs 2 Train Accuracy : 0.77325
Epochs 2 Validation Accuracy : 0.7238
Epochs 3 Train Accuracy : 0.7726
Epochs 3 Validation Accuracy : 0.7534
Epochs 4 Train Accuracy : 0.77815
Epochs 4 Validation Accuracy : 0.7226
Epochs 5 Train Accuracy : 0.764
Epochs 5 Validation Accuracy : 0.7166
Epochs 6 Train Accuracy : 0.7774
Epochs 6 Validation Accuracy : 0.6982
Epochs 7 Train Accuracy : 0.77725
Epochs 7 Validation Accuracy : 0.745
Epochs 8 Train Accuracy : 0.76255
Epochs 8 Validation Accuracy : 0.7452
Epochs 9 Train Accuracy : 0.7871
Epochs 9 Validation Accuracy : 0.7524
Epochs 10 Train Accuracy : 0.7881
Epochs 10 Validation Accuracy : 0.7592
Epochs 11 Train Accuracy : 0.7885
Epochs 11 Validation Accuracy : 0.7578
Epochs 12 Train Accuracy : 0.7693
Epochs 12 Validation Accuracy : 0.7514
Epochs 13 Train Accuracy : 0.7644
Epochs 13 Validation Accuracy : 0

In [ ]:
# 한국어 감성 분류 - LSTM

# 1. 한글을 입력데이터로 사용하기 위해서는 한글을 숫자로 변환!
#    단어사전(한글단어와 숫자가 1:1로 매칭되는 단어사전) 생성해야 해요!
# 2. 단어를 숫자로 변경하기 위해 문장을 특정한 기준으로 잘라내야 해요!
#    특정한 기준으로 분리한 조각을 token
#    영어는 이 token을 만드는게 어렵지 않아요. 공백을 기준으로 짤라서
#    상대적으로 간단한 처리로  token을 구별해 낼 수 있어요!
#    한글은 이  token을 만드는게 쉽지 않아요. 조사가 있어서 그래요!
#    어쨋든 tokenizer라는 도구를 이용해서 token을 구별할 수 있어요!
# 3. 문장의 길이가 다른 경우 당연히 같은 길이로 만들어 줘야 해요!

In [14]:
%reset -f
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = ['영실이는 나를 정말 정말 좋아해',
             '영실이는 영화를 좋아해']

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

# 공백을 기준으로 token들이 생성되고
# 이 token을 이용해서 vacabulary를 생성해요!
print(tokenizer.word_index) # Vacabulary(단어사전)

# 단어사전이 만들어졌으니 이를 이용해서
# 문자열을 숫자의 sequence로 변경할 수 있어요!
word_encoding = tokenizer.texts_to_sequences(sentences)
print(word_encoding)

{'영실이는': 1, '정말': 2, '좋아해': 3, '나를': 4, '영화를': 5}
[[1, 4, 2, 2, 3], [1, 5, 3]]


In [15]:
# 만약 단어사전에 없는 새로운 단어가 등장하면 어떻게 될까요?

%reset -f
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = ['영실이는 나를 정말 정말 좋아해',
             '영실이는 영화를 좋아해']

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

# 공백을 기준으로 token들이 생성되고
# 이 token을 이용해서 vacabulary를 생성해요!
print(tokenizer.word_index) # Vacabulary(단어사전)

# 단어사전이 만들어졌으니 이를 이용해서
# 문자열을 숫자의 sequence로 변경할 수 있어요!
word_encoding = tokenizer.texts_to_sequences(sentences)
print(word_encoding)

new_sentence = ['영실이는 거북이와 나를 좋아해']
new_word_encoding = tokenizer.texts_to_sequences(new_sentence)
print(new_word_encoding)
# [[1, 4, 3]]  Token은 4개지만 encoding된 결과는 3개예요!

# 이런 경우를 위해 특수한 token을 vacabulary에 등록해 줄 수 있어요!
# OOV(Out of vacabulary) token을 등록!~

{'영실이는': 1, '정말': 2, '좋아해': 3, '나를': 4, '영화를': 5}
[[1, 4, 2, 2, 3], [1, 5, 3]]
[[1, 4, 3]]


In [16]:
# 이런 경우를 위해 특수한 token을 vacabulary에 등록해 줄 수 있어요!
# OOV(Out of vacabulary) token을 등록!~

%reset -f
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = ['영실이는 나를 정말 정말 좋아해',
             '영실이는 영화를 좋아해']

tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)

# 공백을 기준으로 token들이 생성되고
# 이 token을 이용해서 vacabulary를 생성해요!
print(tokenizer.word_index) # Vacabulary(단어사전)

# 단어사전이 만들어졌으니 이를 이용해서
# 문자열을 숫자의 sequence로 변경할 수 있어요!
word_encoding = tokenizer.texts_to_sequences(sentences)
print(word_encoding)

new_sentence = ['영실이는 거북이와 나를 좋아해']
new_word_encoding = tokenizer.texts_to_sequences(new_sentence)
print(new_word_encoding)
# [[1, 4, 3]]  Token은 4개지만 encoding된 결과는 3개예요!



{'<OOV>': 1, '영실이는': 2, '정말': 3, '좋아해': 4, '나를': 5, '영화를': 6}
[[2, 5, 3, 3, 4], [2, 6, 4]]
[[2, 1, 5, 4]]


In [18]:
# 추가적으로 문자열 데이터셋에 빈도가 작은 단어가 많이 존재하는 경우
# 이런것들은 제외하는게 좋아요!
# 많이 사용하는 단어들이 있고 적게 사용하는 단어들이 있어요!
# 빈도수가 많은 단어들만 vacabulary에 포함하는게 좋아요!
# num_words 파라미터를 이용해서 token의 개수를 조절할 수 있어요!
%reset -f
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = ['영실이는 나를 정말 정말 좋아해',
             '영실이는 영화를 좋아해']

tokenizer = Tokenizer(oov_token='<OOV>',
                      num_words=3)  # 빈도가 가장 높은 token 3개
tokenizer.fit_on_texts(sentences)

# 공백을 기준으로 token들이 생성되고
# 이 token을 이용해서 vacabulary를 생성해요!
print(tokenizer.word_index) # Vacabulary(단어사전)
{'<OOV>': 1, '영실이는': 2, '정말': 3, '좋아해': 4, '나를': 5, '영화를': 6}

# 단어사전이 만들어졌으니 이를 이용해서
# 문자열을 숫자의 sequence로 변경할 수 있어요!
word_encoding = tokenizer.texts_to_sequences(sentences)
print(word_encoding)

# new_sentence = ['영실이는 거북이와 나를 좋아해']
# new_word_encoding = tokenizer.texts_to_sequences(new_sentence)
# print(new_word_encoding)


{'<OOV>': 1, '영실이는': 2, '정말': 3, '좋아해': 4, '나를': 5, '영화를': 6}
[[2, 1, 1, 1, 1], [2, 1, 1]]


In [20]:
# 마지막으로 pad_sequence도 해 보아요!
%reset -f
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = ['영실이는 나를 정말 정말 좋아해',
             '영실이는 영화를 좋아해']

tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)

# 공백을 기준으로 token들이 생성되고
# 이 token을 이용해서 vacabulary를 생성해요!
print(tokenizer.word_index) # Vacabulary(단어사전)
# {'<OOV>': 1, '영실이는': 2, '정말': 3, '좋아해': 4, '나를': 5, '영화를': 6}

# 단어사전이 만들어졌으니 이를 이용해서
# 문자열을 숫자의 sequence로 변경할 수 있어요!
word_encoding = tokenizer.texts_to_sequences(sentences)
print(word_encoding)

# 문자열의 길이를 똑같이 맞춰줘야 해요!
from tensorflow.keras.preprocessing.sequence import pad_sequences

result = pad_sequences(word_encoding,
                       maxlen=4)
print(result)
# Tokenizer에 대한 기능에 대해 알아보았어요!


{'<OOV>': 1, '영실이는': 2, '정말': 3, '좋아해': 4, '나를': 5, '영화를': 6}
[[2, 5, 3, 3, 4], [2, 6, 4]]
[[5 3 3 4]
 [0 2 6 4]]
